# Embedding Positions
When the transformer reads in tokens, it does so all at once (a nod to its parallel processing). However, this means that tokesn have no inherent ordering - we need to give them some so that "the cat sat" has a totally different meaning to "cat sat the".

Without positional information, the attention mechanism computes the same output for "cat sat the" as it does "the cat sat" because attention is just weighted sums over embeddings - it does not care about order. Both sentences have identical tokens, so the model sees them as identical inputs.

### Absolute Positional Embeddings
**Positional Embeddings**

In nanoGPT, Karpathy uses positional embeddings, which are simply another embedding table indexed by position instead of token. A large limitation here is that we are unable to encode positions beyond what we saw in training.

**Sinusoidal Embeddings**

In 'Attention is all you Need, the authors use sinusoidal embeddings: fixed sine/cosine functions for every position. These work for any length because of their sequential nature, but are weaker at relative positions:

$PE(pos, 2i) = sin(\frac{pos}{10000^{(2i/embeddings dim)}})$

$PE(pos, 2i+1) = cos(\frac{pos}{10000^{(2i+1/embeddings dim)}})$

where position is the position in the sequence and i is the dimension index (0,1,2,...embeddings_dim/2). In this way, every position gets a unique fingerprint - no two rows are identical. We also observe fast waves in earlir dimensions and slower in later dimensions, suggesting that each frequency captures a different 'scale' of the position (like days, hours, mins, seconds on a clock).

In positional embeddings, every positional embedding is essentially independent of each other.  But in reality we might expect that positions 1 and 2 are more similar than 1 and 512.

### RoPE: Rotary Position Embeddings
Zooming out, we want to encode positions because it gives attention additional context behind the meaning of a word or token. In the methods described above, the way we had added positional information was by adding additional dimensions in a given token's embeddings. However, the core piece of information we are trying to convey when communicating meaning is where one token is relative to another. For instance, when we think about the sentence: "the dog chased the squirrel", it does not matter if "squirrel" is in position 4 or if it is in position 184 if we know that "dog" is 3 positions ahead of it for the purpose of language modeling. Of course we lose some amount of meaning when removing absolute position, but for the primary signal, relying on relative positon gets us most of the way there. Empirical evidence backs this up: models using RoPE outperform sinusoidal models.

We can imagine a 2d plane where the word 'dog' is represented by some vector. If the word 'dog' appears in the second position, we might see that vector rotated clockwise by $\theta$ degrees. If it appears even later, we might see that vector rotated clockwise by the integer multiple of its position in the sentence (so maybe by $4*\theta$ degrees).

Advantages:
* can handle increased length / caching (adding words to end of sentence does not affect words at the beginning of a sentence)
* relative positions preserved ("the cat sat" vs. "Yesterday morning, the cat sat": still only one token apart). Mathematically, the dot product remain unchanged.

Skipping the mathematical portion since I did it mostly by hand. Note: add later.


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
class RoPE(nn.Module):
  def __init__(self, embeddings_dim, max_seq_len: int = 2048, theta: float = 10000.):
    super().__init__()

    # asserting model is even (0,1), (2,) -> this would be a problem
    assert(embeddings_dim % 2 == 0)

    # indices of the first dimension of each pair: 0, 2, 4, 6,...
    pair_number = torch.arange(0, (embeddings_dim // 2), 1).float()

    # compute frequencies, one for each pair
    denom = theta ** (2*pair_number / embeddings_dim)
    inv_freq = 1. / denom

    # compute position indices
    positions = torch.arange(0, max_seq_len, 1)

    # compute rotation angles for each position and each pair
    pair_angles = torch.outer(positions, inv_freq)

    # compute rotation angles for each position and each dimension
    # note the ordering: [0,0,1,1,2,2...]
    angles = torch.repeat_interleave(pair_angles, repeats=2, dim=-1)

    # compute sine and cosine
    sine = torch.sin(angles)
    cosine = torch.cos(angles)
    self.register_buffer("sine_cached", sine)
    self.register_buffer("cos_cached", cosine)

  @staticmethod
  def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    For a given pair (x0, x1), the rotation formula is as follows:
    x0' = x0*cos(theta) - x1*sin(theta)
    x1' = x0*sin(theta) + x1*cos(theta)
    We could loop through all pairs, but that would be inefficient.
    rotate_half = [-x1, x0, -x3, x2, -x5, x4,...]

    x' = x*cos(theta) + rotate_half(x)*sin(theta)

    x0' = x0*cos(theta) - x1*sin(theta)
    x1' = x0*sin(theta) + x1*cos(theta)
    x2' = x2*cos(theta) - x3*sin(theta)
    x3' = x2*sin(theta) + x3*cos(theta)
    """
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    rotated = torch.stack((-x_odd, x_even), dim=-1).flatten(start_dim=-2)
    return rotated

# now appling ROPE to a given Q or K vector
  def forward(self, x: torch.Tensor) -> torch.Tensor:

    # [batch, heads, seq, dim_per_head]
    seq_len = x.shape[-2]

    # get cos and sine
    cos = self.cos_cached[:seq_len]
    sin = self.sine_cached[:seq_len]

    # broadcast
    # each position and each dimension gets its own RoPE angle,
    # but the same angle table is shared across batches and heads.
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)

    x_prime = x * cos + self.rotate_half(x) * sin
    return x_prime

In [4]:
x = torch.tensor([1., 2., 3., 4.])
rope = RoPE(8)
rope.rotate_half(x)

tensor([-2.,  1., -4.,  3.])

In [6]:
rope = RoPE(8)
q = torch.randn(1, 1, 4, 8)
out = rope(q)
print(out.shape)

torch.Size([1, 1, 4, 8])
